In [ ]:
import pandas as pd

# ============================================================
# BƯỚC 0 — Đọc dữ liệu
# ============================================================
df = pd.read_excel('../data/raw/TTTH_THPT_Cleaned.xlsx')
print(f"Số dòng ban đầu: {len(df)}")

# ============================================================
# BƯỚC 1 — Loại dòng không trúng tuyển (KTT)
# ============================================================
df = df[df['KQ'] == 'TT'].copy()
print(f"Sau khi loại KTT: {len(df)}")

# ============================================================
# BƯỚC 2 — Loại tổ hợp D10 (chưa xác nhận được mapping môn)
# ============================================================
df = df[df['Mã tổ hợp trúng tuyển'] != 'D10'].copy()
print(f"Sau khi loại D10: {len(df)}")

# ============================================================
# BƯỚC 3 — Remap 3 mã ngành cũ sang mã mới, có kiểm tra tổ hợp hợp lệ
# ============================================================
# (mã cũ -> (mã mới, tập tổ hợp hợp lệ theo bảng 39 ngành))
remap_rules = {
    7520311: (7510401, {'B00', 'B08', 'A00', 'D07'}),  # Kỹ thuật hóa phân tích -> CN kỹ thuật hóa học
    7620303: (7540105, {'B00', 'B08', 'A00', 'D07'}),  # Khoa học thủy sản -> CN chế biến thủy sản
    7540110: (7540106, {'B00', 'B08', 'A00', 'D07'}),  # Đảm bảo CL&ATTP cũ -> mới
}

rows_to_drop = []
for ma_cu, (ma_moi, tohop_hople) in remap_rules.items():
    mask = df['Mã ngành trúng tuyển'] == ma_cu
    # Dòng tổ hợp KHÔNG hợp lệ với ngành mới -> đánh dấu loại bỏ
    invalid_mask = mask & (~df['Mã tổ hợp trúng tuyển'].isin(tohop_hople))
    rows_to_drop.extend(df[invalid_mask].index.tolist())
    # Dòng tổ hợp hợp lệ -> remap sang mã mới
    valid_mask = mask & (df['Mã tổ hợp trúng tuyển'].isin(tohop_hople))
    df.loc[valid_mask, 'Mã ngành trúng tuyển'] = ma_moi

df = df.drop(index=rows_to_drop)
print(f"Sau khi remap 3 mã ngành (loại {len(rows_to_drop)} dòng lệch tổ hợp): {len(df)}")

# ============================================================
# BƯỚC 4 — Loại các mã ngành còn lại ngoài phạm vi 39 ngành
# ============================================================
danh_sach_39_nganh = {
    7810103, 7810201, 7810202, 7819009, 7819010, 7810101, 7380101, 7380107,
    7220201, 7220204, 7480201, 7480202, 7460108, 7340301, 7340201, 7340205,
    7340115, 7340122, 7510605, 7340101, 7340120, 7540204, 7340123, 7540101,
    7540106, 7340129, 7540105, 7510202, 7510203, 7520115, 7510301, 7510303,
    7510401, 7510402, 7420201, 7850101, 7510406, 7480107, 7510601,
}

before = len(df)
df = df[df['Mã ngành trúng tuyển'].isin(danh_sach_39_nganh)].copy()
print(f"Sau khi loại ngành ngoài phạm vi ({before - len(df)} dòng bị loại): {len(df)}")

# ============================================================
# BƯỚC 5 — Flatten M1/M2/M3 theo đúng thứ tự môn của từng tổ hợp
# ============================================================
to_hop_map = {
    'A00': ['Toan', 'Ly', 'Hoa'],
    'A01': ['Toan', 'Ly', 'Anh'],
    'B00': ['Toan', 'Hoa', 'Sinh'],
    'B08': ['Toan', 'Sinh', 'Anh'],
    'C00': ['Van', 'Su', 'Dia'],
    'C01': ['Van', 'Toan', 'Ly'],
    'C02': ['Van', 'Toan', 'Hoa'],
    'C03': ['Van', 'Toan', 'Su'],
    'D01': ['Toan', 'Van', 'Anh'],
    'D07': ['Toan', 'Hoa', 'Anh'],
    'D09': ['Toan', 'Su', 'Anh'],
    'D14': ['Van', 'Anh', 'Su'],
    'D15': ['Van', 'Dia', 'Anh'],
    'X01': ['Toan', 'Van', 'Gdktpl'],
    'X26': ['Toan', 'Tin', 'Anh'],
}

mon_columns = ['Toan', 'Ly', 'Hoa', 'Anh', 'Van', 'Su', 'Sinh', 'Dia', 'Gdktpl', 'Tin']
for mon in mon_columns:
    df[f'diem_{mon}'] = pd.NA

def flatten_row(row):
    to_hop = row['Mã tổ hợp trúng tuyển']
    mon_list = to_hop_map.get(to_hop)
    if mon_list is None:
        return row
    row[f'diem_{mon_list[0]}'] = row['M1']
    row[f'diem_{mon_list[1]}'] = row['M2']
    row[f'diem_{mon_list[2]}'] = row['M3']
    return row

df = df.apply(flatten_row, axis=1)

# ============================================================
# BƯỚC 6 — Chọn cột cuối cùng, đổi tên cho khớp file khảo sát
# ============================================================
final_cols = ['Mã ngành trúng tuyển', 'Mã tổ hợp trúng tuyển'] + [f'diem_{m}' for m in mon_columns]
df_final = df[final_cols].copy()
df_final = df_final.rename(columns={
    'Mã ngành trúng tuyển': 'ma_nganh',
    'Mã tổ hợp trúng tuyển': 'to_hop_thi',
})

print(f"\nSố dòng cuối cùng: {len(df_final)}")
print(df_final.head())

# ============================================================
# BƯỚC 7 — Xuất file
# ============================================================
df_final.to_csv('../data/processed/TTTH_processed_flatten.csv', index=False, encoding='utf-8-sig')
print("\nĐã lưu: TTTH_processed_flatten.csv")